# Install Library

In [1]:
!pip install transformers
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 14.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.12.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which i

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Import Library

In [3]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer
from google.colab import drive

# Connect to Drive

In [4]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
# Path folder tempat file CSV berada
combined_df = pd.read_csv('/content/drive/MyDrive/File/CODES/Summarize/6_Kapan_Lagi_clickbait/berlebihan/pre_berlebihan.csv')

combined_df.head()

,title,category,publish_date,article_url,content,label
0,Wanita dengan Tubuh Berisi Terbukti Bikin Pria...,Lifestyle,2019-09-16,https://lifestyle.kompas.com/read/2019/09/16/2...,"Berbagai usaha dilakukan, mulai dari olahraga ...",clickbait
1,Kronologi Lengkap Pemuda di Bandung Tusuk Gadi...,News,2019-10-09,https://regional.kompas.com/read/2019/09/10/23...,"RG (22), pemuda yang nekat menusuk Z (16), sis...",clickbait
2,"Sarapan Bikin Ngantuk, Mitos atau Fakta?",Lifestyle,2019-09-16,https://lifestyle.kompas.com/read/2019/09/16/2...,"Tak sedikit orang yang beranggapan, bahwa , p...",clickbait
3,Puisi Hairdryer Dian Sastro yang Tak Terduga d...,Entertainment,2019-10-09,https://entertainment.kompas.com/read/2019/09/...,"Puisi , karya artis peran , ramai diperbinca...",clickbait
4,"Unggah Foto Bersama Habibie, Syahrini: Semoga ...",Entertainment,2019-11-09,https://entertainment.kompas.com/read/2019/09/...,"Penyanyi , turut menyampaikan duka atas menin...",clickbait


In [7]:
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3789 entries, 0 to 3788
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   title         3789 non-null   object
 1   category      3789 non-null   object
 2   publish_date  3789 non-null   object
 3   article_url   3789 non-null   object
 4   content       3787 non-null   object
 5   label         3789 non-null   object
dtypes: object(6)
memory usage: 177.7+ KB


In [10]:
nan_summary = combined_df[combined_df['content'].isna()]


nan_summary

,title,category,publish_date,article_url,content,label
2516,"Tinggalkan Google, Wanita Ini Ungkap Rahasia R...",tekno,2019-09-18,https://tekno.tempo.co/read/1249203/tinggalkan...,NaN,clickbait
2603,"Robot Militer Otonom Diklaim Berbahaya, Apa Saja?",tekno,2019-09-18,https://tekno.tempo.co/read/1249237/robot-mili...,NaN,clickbait


In [11]:
combined_df=combined_df.dropna()

# Duplicate


In [12]:
data = combined_df.copy()

In [13]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3787 entries, 0 to 3788
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   title         3787 non-null   object
 1   category      3787 non-null   object
 2   publish_date  3787 non-null   object
 3   article_url   3787 non-null   object
 4   content       3787 non-null   object
 5   label         3787 non-null   object
dtypes: object(6)
memory usage: 207.1+ KB


# Tokenize

Mencoba kode lain agar tokenize lebih mudah di pahami

# Word Embedding

In [14]:
from transformers import AutoTokenizer, AutoModel
import torch

# Memuat IndoBERT dan tokenizer
tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p1", use_fast=True)
model = AutoModel.from_pretrained("indobenchmark/indobert-base-p1")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

In [16]:
# Fungsi untuk mendapatkan embedding dari IndoBERT
def get_embeddings(input_ids):
    with torch.no_grad():
        outputs = model(input_ids=input_ids)
    # Mengambil hidden states dari layer terakhir dan rata-rata embeddingsnya
    embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings

# Fungsi untuk mengonversi tokenized data ke dalam bentuk input_ids dan mendapatkan embedding
def get_input_ids_and_embeddings(data_column):
    input_ids_list = []
    embeddings_list = []

    for text in data_column:
        # Tokenisasi
        tokens = tokenizer(text, padding='max_length', max_length=256, truncation=True, return_tensors="pt")
        input_ids = tokens['input_ids']  # Ambil input_ids
        input_ids_list.append(input_ids)

        # Mendapatkan embeddings
        embedding = get_embeddings(input_ids)
        embeddings_list.append(embedding)

    return input_ids_list, embeddings_list

# Dapatkan input_ids dan embeddings untuk title dan summary
title_input_ids, title_embeddings = get_input_ids_and_embeddings(data['title'])
summary_input_ids, summary_embeddings = get_input_ids_and_embeddings(data['content'])


## Mengubah embeddings menjadi array numpy

In [17]:
# Mengubah embeddings menjadi array numpy agar bisa disimpan dalam DataFrame
title_embeddings_array = np.vstack([embedding.numpy() for embedding in title_embeddings])
summary_embeddings_array = np.vstack([embedding.numpy() for embedding in summary_embeddings])

# Membuat DataFrame baru dengan embeddings numerik
data_embeddings = pd.DataFrame({
    'title_embeddings': list(title_embeddings_array),
    'summary_embeddings': list(summary_embeddings_array)
})

# Cosine Similarity

In [18]:
# hitung cosine similarity pada matrix berita
cosine_sim = cosine_similarity(title_embeddings_array, summary_embeddings_array)
print(cosine_sim)

[[0.11928782 0.13128646 0.09800425 ... 0.2453731  0.18388554 0.17838743]
 [0.11780906 0.13218574 0.09750348 ... 0.24568364 0.18407705 0.17838702]
 [0.12166087 0.13311192 0.10015878 ... 0.24583969 0.18585397 0.18088758]
 ...
 [0.11826164 0.13033669 0.097785   ... 0.24760845 0.18356365 0.17760554]
 [0.11711738 0.12720458 0.09591059 ... 0.2458737  0.18277028 0.17562726]
 [0.11823764 0.13230564 0.09884524 ... 0.24757928 0.18535931 0.18107027]]


In [19]:
# membuat dataframe dari varible cosine similarity dengan baris dan kolom title
cosine_sim_df = pd.DataFrame(cosine_sim, index=data['content'], columns=data['title'])

print(cosine_sim_df.shape)

(3787, 3787)


## Check Similarity Result

In [20]:
# Menampilkan pasangan similarity antara 'title' dan 'summary' yang sesuai dengan urutan
for index, row in cosine_sim_df.iterrows():
    # The index is a summary, and you need to find the corresponding title
    # to access the similarity value. However, since you have both summaries
    # and titles as indices, you can access the similarity directly using
    # the index and the corresponding title from the 'data' DataFrame.

    title = data.loc[data['content'] == index, 'title'].iloc[0] # Get the title for the current summary
    similarity_value = cosine_sim_df.loc[index, title]  # Access the similarity using both summary and title

    print(f"Title: {title}")
    print(f"Summary: {index}")
    print(f"Cosine Similarity: {similarity_value}\n")
    print("-" * 50)  # Pemisah antar pasangan

Streaming output truncated to the last 5000 lines.
Cosine Similarity: 0.17044758796691895

--------------------------------------------------
Title: 5 Makanan Khas Keraton Yogyakarta yang Legendaris
Summary: BERKUNJUNG ke Yogyakarta rasanya belum lengkap tanpa mencicipi hidangan khas Kerajaan,. Makanan khas kerajaan Keraton Yogyakarta ini merupakan hidangan spesial untuk raja-raja Yogyakarta sejak ratusan tahun yang lalu.,Buat pelancong pasti penasaran dengan makanan yang disajikan untuk Sri Sultan? Dimasak oleh para abdi dalem di pawon atau dapur khusus yang berada di dalam keraton, penyajian makanan khas Kerajaan,untuk Sri Sultan ini punya tata caranya sendiri yang disebut tata cara ladosan dhahar dalem.,,Kira-kira seperti apa selera makanan para raja di Keraton Yogyakarta? Berikut ulasan selengkapnya seperti dirangkum dari,:,,Makanan Khas Kerajaan Keraton Yogyakarta yang satu ini sejenis salad versi Jawa. Berisi kombinasi antara selada, mentimun, tomat dan telur rebus.,Untuk siraman

In [23]:
# List to store the results
results = []

# Menampilkan pasangan similarity antara 'title' dan 'summary' yang sesuai dengan urutan
for index, row in cosine_sim_df.iterrows():
    # Mendapatkan judul yang sesuai dengan summary berdasarkan index
    title = data.loc[data['content'] == index, 'title'].iloc[0]
    similarity_value = cosine_sim_df.loc[index, title]  # Mengakses nilai similarity

    # Menyimpan pasangan dan nilai similarity ke dalam list
    results.append([title, index, similarity_value])

# Membuat DataFrame untuk hasilnya
similarity_df = pd.DataFrame(results, columns=['Title', 'Summary', 'Cosine Similarity'])


In [24]:
similarity_df.head()

,Title,Summary,Cosine Similarity
0,Wanita dengan Tubuh Berisi Terbukti Bikin Pria...,"Berbagai usaha dilakukan, mulai dari olahraga ...",0.119288
1,Kronologi Lengkap Pemuda di Bandung Tusuk Gadi...,"RG (22), pemuda yang nekat menusuk Z (16), sis...",0.132186
2,"Sarapan Bikin Ngantuk, Mitos atau Fakta?","Tak sedikit orang yang beranggapan, bahwa , p...",0.100159
3,Puisi Hairdryer Dian Sastro yang Tak Terduga d...,"Puisi , karya artis peran , ramai diperbinca...",0.632686
4,"Unggah Foto Bersama Habibie, Syahrini: Semoga ...","Penyanyi , turut menyampaikan duka atas menin...",0.144811


## Save Similarity

In [25]:
# Menyimpan DataFrame ke file CSV di lokasi yang ditentukan
similarity_df.to_csv('/content/drive/MyDrive/File/CODES/Similarity/6_Similarity_Berlebihan_Tambahan_Base1/similarity_berlebihan_tambah_base1.csv', index=False)